In [1]:
import numpy as np
import cv2
import os
import random
import matplotlib.pyplot as plt

# **Data - Extended Cohn-Kanade**


In [2]:
def find_Class(directory):
    try:
        folders = [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f))]
        return folders
    except FileNotFoundError:
        raise ValueError(f"Directory not found: {directory}")

DIRECTORY= r"/content/drive/MyDrive/CK Dataset"
CATAGORIES= []
try:
    folders = find_Class(DIRECTORY)
    print(f"Directories in '{DIRECTORY}':")
    for folder in folders:
        CATAGORIES.append(folder)
except ValueError as e:
    print(e)

CATAGORIES

Directories in '/content/drive/MyDrive/CK Dataset':


['sadness', 'happy', 'disgust', 'surprise', 'anger', 'contempt', 'fear']

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
data=[]

for categories in CATAGORIES:
    folder=os.path.join(DIRECTORY,categories)
    label=CATAGORIES.index(categories)


    for img in os.listdir(folder):
        img=os.path.join(folder,img)
        img_arr=cv2.imread(img)
        if img_arr is not None:  # Check if the image is successfully loaded
            img_arr = cv2.resize(img_arr, (100, 100))
            data.append([img_arr, label])
        else:
            print(f"Failed to load image {img}")

In [6]:
len(data)

981

In [7]:
random.shuffle(data)

In [8]:
x=[]
y=[]


for features,label in data:
    x.append(features)
    y.append(label)
X= np.array(x)
Y=np.array(y)

In [9]:
X=X/255

In [10]:
X.shape
Y.shape

(981,)

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Activation
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# **Model Architecture 0.001**

In [12]:
model=Sequential()
model.add( Conv2D(64,(3,3),input_shape=X.shape[1:],activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add( Conv2D(32,(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add( Conv2D(32,(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))


model.add(Flatten())

model.add(Dense(7,activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
checkpoint=ModelCheckpoint(r'fer.keras',
                          monitor='val_loss',
                          mode='min',
                          save_best_only=True,
                          verbose=1)
earlystop=EarlyStopping(monitor='val_loss',
                        mode='min',
                       min_delta=0.001,
                       patience=20,
                       verbose=1,
                       restore_best_weights=True)

callbacks=[checkpoint,earlystop]

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.fit(X,Y,epochs=50,validation_split=0.20,callbacks =callbacks)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 672ms/step - accuracy: 0.2311 - loss: 1.8697
Epoch 1: val_loss improved from inf to 1.75168, saving model to fer.keras
25/25 ━━━━━━━━━━━━━━━━━━━━ 19s 723ms/step - accuracy: 0.2317 - loss: 1.8686 - val_accuracy: 0.2589 - val_loss: 1.7517
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 704ms/step - accuracy: 0.3292 - loss: 1.7443
Epoch 2: val_loss improved from 1.75168 to 1.37318, saving model to fer.keras
25/25 ━━━━━━━━━━━━━━━━━━━━ 22s 773ms/step - accuracy: 0.3320 - loss: 1.7422 - val_accuracy: 0.5279 - val_loss: 1.3732
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 671ms/step - accuracy: 0.5924 - loss: 1.2807
Epoch 3: val_loss improved from 1.37318 to 0.65498, saving model to fer.keras
25/25 ━━━━━━━━━━━━━━━━━━━━ 18s 723ms/step - accuracy: 0.5950 - loss: 1.2722 - val_accuracy: 0.7868 - val_loss: 0.6550
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 683ms/step - accuracy: 0.7814 - loss: 0.6263
Epoch 4: val_loss improved from 0.65498 to 0.38906, saving model to fer.

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 98, 98, 64)          │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 49, 49, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 47, 47, 32)          │          18,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 23, 23, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 21, 21, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 10, 10, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 3200)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 7)                   │          22,407 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 155,735 (608.34 KB)

 Trainable params: 51,911 (202.78 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 103,824 (405.57 KB)

In [16]:
model.save('ck_model.keras')

In [17]:
from flask import Flask, request, jsonify
import tensorflow as tf
import cv2
import numpy as np

In [19]:
model = tf.keras.models.load_model('ck_model.keras')

/usr/local/lib/python3.10/dist-packages/keras/src/saving/saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [20]:
app = Flask(__name__)

In [21]:
@app.route('/predict', methods=['POST'])
def predict():
    image_data = request.files['image'].read()
    np_array = np.frombuffer(image_data, np.uint8)
    image = cv2.imdecode(np_array, cv2.IMREAD_COLOR)

    image = cv2.resize(image, (224, 224))
    image = image / 255.0
    image = np.expand_dims(image, axis=0)

    prediction = model.predict(image)
    emotion = np.argmax(prediction)

    return jsonify({'emotion': emotion})

In [22]:
if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat
